In [13]:
import faiss
import pickle
import numpy as np

from sentence_transformers import SentenceTransformer
from langchain_community.llms import Ollama

In [14]:
index = faiss.read_index("../vector_db/bank_index.faiss")

print("FAISS index loaded successfully!")
print("Vectors stored:", index.ntotal)

FAISS index loaded successfully!
Vectors stored: 66


In [15]:
with open("../vector_db/chunks.pkl", "rb") as f:
    chunks = pickle.load(f)

with open("../vector_db/chunk_metadata.pkl", "rb") as f:
    chunk_metadata = pickle.load(f)

print("Chunks loaded:", len(chunks))

Chunks loaded: 66


In [16]:
model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded!


In [17]:
llm = Ollama(model="llama3.1")

print("Llama connected!")

Llama connected!


In [18]:
query = """
Customer identity verification documents required before opening a new bank account.
"""

In [19]:
query_embedding = model.encode([query])
query_embedding = np.array(query_embedding).astype("float32")

In [20]:
distances, indices = index.search(query_embedding, k=5)

In [21]:
print("Retrieved Documents:\n")

for idx in indices[0]:
    print(chunk_metadata[idx])

Retrieved Documents:

KYC_Policy.pdf
KYC_Policy.pdf
KYC_Policy.pdf
Internal_Control_Library.pdf
Information_Security_Policy.pdf


In [22]:
context = "\n\n".join([chunks[idx] for idx in indices[0]])

In [23]:
prompt = f"""
You are an AI Banking Compliance Assistant.

Use ONLY the information provided in the context.

If the answer cannot be found, say:
"The provided documents do not contain sufficient information."

Context:
-------------------------
{context}
-------------------------

Question:
{query}

Answer professionally in 5-8 sentences.
"""

In [24]:
response = llm.invoke(prompt)

In [26]:
print(response)

Based on the provided documentation, specifically the Customer Identification Procedure section of the Know Your Customer (KYC) Policy, the following Officially Valid Documents (OVDs) are required for customer identity verification prior to opening a new bank account:

* Aadhaar
* Passport
* Voter ID
* Driving Licence
* PAN or Form 60

Additionally, one proof of address document is also required, such as an Aadhaar or utility bill not older than two months. The Bank shall undertake the Customer Identification Procedure (CIP) for every new relationship prior to account opening, including verification through Aadhaar e-KYC, offline Aadhaar XML, digital KYC, or physical document verification, as applicable.
